In [25]:
import pandas as pd
import json
import ollama
#from pydantic import BaseModel, Field
import json
from db.pgsql import pgsql
import ast

from datetime import datetime, timedelta, timezone
#from getpass import getpass
import keyring
from pathlib import Path
import os
import time

import requests

In [26]:
ENDPOINT = "https://data.alpaca.markets/v1beta1/news"
PAGE_SIZE = 50
SYMBOLS_PER_REQUEST = 40
LOOKBACK = timedelta(hours=24)

api_key = os.getenv("ALPACA_API_KEY") or keyring.get_password(
    "api.alpaca.markets", "ALPACA_API_KEY")

secret_key = os.getenv("ALPACA_SECRET_KEY") or keyring.get_password(
    "api.alpaca.markets", "ALPACA_SECRET_KEY")


ticker_file = next(
    (
        parent / "documentation" / "alpaca_tickers.txt"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "documentation" / "alpaca_tickers.txt").is_file()
    ),
    None,
)
if ticker_file is None:
    raise FileNotFoundError(
        "Could not find documentation/alpaca_tickers.txt from the notebook's working directory."
    )

symbols = list(
    dict.fromkeys(
        line.strip().upper()
        for line in ticker_file.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    )
)
if not symbols:
    raise ValueError(f"No ticker symbols found in {ticker_file}")

#api_key = os.getenv("ALPACA_API_KEY") or getpass("Alpaca API key: ")
#secret_key = os.getenv("ALPACA_SECRET_KEY") or getpass("Alpaca secret key: ")
def get_api_key_secret():
    api_key = os.getenv("ALPACA_API_KEY") or keyring.get_password(
        "api.alpaca.markets", "ALPACA_API_KEY")
    secret_key = os.getenv("ALPACA_SECRET_KEY") or keyring.get_password(
        "api.alpaca.markets", "ALPACA_SECRET_KEY")
    if not api_key or not secret_key:
        raise ValueError("Both Alpaca API credentials are required.")
    return api_key, secret_key

def get_ticker_file(folder, file_name):
    ticker_file = next(
        (
            parent / folder / file_name
            for parent in (Path.cwd(), *Path.cwd().parents)
            if (parent / folder / file_name).is_file()
        ),
        None,
    )
    if ticker_file is None:
        raise FileNotFoundError(
            f"Could not find {folder}/{file_name} from the notebook's working directory."
        )
    return ticker_file

ticker_file = get_ticker_file("documentation", "alpaca_tickers.txt")

def get_symbols_from_file(ticker_file):
    return list(
        dict.fromkeys(
            line.strip().upper()
            for line in ticker_file.read_text(encoding="utf-8").splitlines()
            if line.strip() and not line.lstrip().startswith("#")
        )
    )

symbols = get_symbols_from_file(ticker_file)


api_key, secret_key = get_api_key_secret()
headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": api_key,
    "APCA-API-SECRET-KEY": secret_key,
}

ENDPOINT = "https://data.alpaca.markets/v1beta1/news"
PAGE_SIZE = 50
SYMBOLS_PER_REQUEST = 40
LOOKBACK = timedelta(hours=24)

end_time = datetime.now(timezone.utc)
start_time = end_time - LOOKBACK


print(f"Loaded {len(symbols)} symbols from {ticker_file}")
print(f"News interval (UTC): {start_time:%Y-%m-%d %H:%M:%S} to {end_time:%Y-%m-%d %H:%M:%S}")

Loaded 515 symbols from /home/djmead/Documents/Projects/Financial/documentation/alpaca_tickers.txt
News interval (UTC): 2026-10-15 01:39:15 to 2026-10-16 01:39:15


In [27]:
def to_rfc3339(value):
    return value.isoformat(timespec="seconds").replace("+00:00", "Z")

def retrieve_articles(symbols):
    news_by_id = {}
    request_count = 0

    for offset in range(0, len(symbols), SYMBOLS_PER_REQUEST):
        symbol_batch = symbols[offset : offset + SYMBOLS_PER_REQUEST]
        page_token = None

        while True:
            params = {
                "symbols": ",".join(symbol_batch),
                "start": to_rfc3339(start_time),
                "end": to_rfc3339(end_time),
                "sort": "desc",
                "limit": PAGE_SIZE,
                "include_content": "false",
            }
            if page_token:
                params["page_token"] = page_token

            response = requests.get(ENDPOINT, headers=headers, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or not isinstance(payload.get("news"), list):
                raise ValueError("Unexpected Alpaca news response: expected an object with a 'news' list.")

            request_count += 1
            for article in payload["news"]:
                article_id = article.get("id")
                if article_id is None:
                    raise ValueError("Alpaca returned a news article without an ID.")

                if article_id not in news_by_id:
                    news_by_id[article_id] = article
                else:
                    known_symbols = set(news_by_id[article_id].get("symbols") or [])
                    known_symbols.update(article.get("symbols") or [])
                    news_by_id[article_id]["symbols"] = sorted(known_symbols)

            page_token = payload.get("next_page_token")
            if not page_token:
                break
            time.sleep(0.7)

        if offset + SYMBOLS_PER_REQUEST < len(symbols):
            time.sleep(0.7)

    articles = list(news_by_id.values())
    print(f"Received {len(articles)} unique articles in {request_count} API requests.")
    return articles





In [28]:
articles = retrieve_articles(symbols)
type(articles[0]['symbols'])

Received 372 unique articles in 18 API requests.


list

In [29]:
def fix_news_list_from_raw(articles: list[dict]) -> list[dict]:
    with open('../../tests/assets_us_equity.json', 'r') as file:
        assets_us_equity = json.load(file)
    assets_us_equity_df = pd.DataFrame(assets_us_equity)
    
    for news in articles:
        tickers = news.get('symbols', [])
        company_dict = {}
        for ticker in tickers:
            try:
                name = assets_us_equity_df.loc[assets_us_equity_df['symbol'] == ticker, 'name'].values[0]
            except IndexError:
                name = None
            company_dict[ticker] = name
        message = "HEADLINE: " + str(news.get('headline', '')) + "\nSUMMARY: " + str(news.get('summary', ''))  # if summary is not nan then message is summary, else message is headline
        while "\n\n" in message:
            message = message.replace("\n\n", "\n")

        news['message'] = message
        news['company_names'] = company_dict
    return articles

In [30]:

#with open('../../tests/assets_us_equity.json', 'r') as file:
#    assets_us_equity = json.load(file)
#assets_us_equity_df = pd.DataFrame(assets_us_equity)

def fix_news_list(news_list):
    with open('../../tests/assets_us_equity.json', 'r') as file:
        assets_us_equity = json.load(file)
    assets_us_equity_df = pd.DataFrame(assets_us_equity)
    
    for news in news_list:
        tickers = ast.literal_eval(news.get('tickers', []))
        company_dict = {}
        for ticker in tickers:
            try:
                name = assets_us_equity_df.loc[assets_us_equity_df['symbol'] == ticker, 'name'].values[0]
            except IndexError:
                name = None
            company_dict[ticker] = name
        message = "HEADLINE: " + str(news.get('headline', '')) + "\nSUMMARY: " + str(news.get('summary', ''))  # if summary is not nan then message is summary, else message is headline
        while "\n\n" in message:
            message = message.replace("\n\n", "\n")

        news['message'] = message
        news['company_names'] = company_dict
    return news_list


def build_scores_schema(tickers) -> dict:
    """JSON schema requiring one score in [-1, 1] for every ticker."""
    return {
        'type': 'object',
        'properties': {
            t: {'type': 'number', 'minimum': -1.0, 'maximum': 1.0} for t in tickers
        },
        'required': list(tickers),
        'additionalProperties': False,
    }


class HeadlineSentimentAnalyzer:
    def __init__(
        self,
        model: str = 'qwen3:8b',
        keep_alive: str = '10m',
        system_prompt_in_model: bool = False,
        temperature: float = 0.0,
    ):
        self.model = model
        self.keep_alive = keep_alive
        self.system_prompt_in_model = system_prompt_in_model
        self.temperature = temperature
        #self.system_prompt = (
        #    "You are an expert financial analyst. Your task is to look at a news message "
        #    "and judge its sentiment regarding each of the given publicly traded companies. "
        #    "You will be given a JSON dictionary (company_names) that matches ticker symbols to company names, "
        #    "and the news message. "
        #    "Provide a rating score for every ticker strictly between -1.0 (very bearish/negative) and "
        #    "+1.0 (very bullish/positive). Highly neutral or purely factual announcements "
        #    "without clear financial impact, or companies not meaningfully discussed, should be given 0.0. "
        #    "Return a JSON object with the ticker symbols as keys and their sentiment scores as values."
        #)
        self.system_prompt = ("""You are an expert financial analyst. 
            Your task is to analyze a news message and judge its sentiment regarding specific publicly traded companies. 

            INPUTS PROVIDED:
            1. `company_names`: A JSON dictionary mapping ticker symbols to company names.
            2. `news_message`: The text text to be analyzed.

            SCORING CRITERIA:
            Assign a sentiment score strictly between -1.0 and +1.0 for EVERY ticker provided in the `company_names` dictionary.
            * Scores can be a decimal from -1 to +1 such as -0.75, 0.31, 0.89, etc.
            * +1.0 = Very Bullish / Strongly Positive (e.g., massive earnings beat, major acquisition, large new government contract)
            * +0.5 = Moderately Bullish / Positive (e.g., incremental contract win, steady growth)
            *  0.0 = Neutral, purely factual announcements without immediate financial impact, mixed news, or if the company is merely mentioned without meaningful discussion.
            * -0.5 = Moderately Bearish / Negative (e.g., slight earnings miss, executive departure)
            * -1.0 = Very Bearish / Strongly Negative (e.g., severe regulatory fines, catastrophic fraud, loss of government contract)
            #* These are just examples. If you think it is appropriate to assign a different score, from -1 to +1, based on the context, you may do so.

            STRICT OUTPUT CONSTRAINTS:
            - You must return ONLY a raw JSON object where keys are the exact ticker symbols from `company_names` and values are the numerical sentiment scores (floats).
            - Do not include markdown blocks like ```json ... ```.
            - Do not include any introductory text, explanations, or postscript commentary. 
            - If a ticker from `company_names` is not mentioned or relevant to the text, you must still include it in the JSON object with a score of 0.0.

            OUTPUT TEMPLATE:
            {
            "TICKER1": 0.0,
            "TICKER2": 0.8,
            "TICKER3": -0.27,
            }
            """)
        self.custom_model = 'fin_news_analyzer'
        ollama.create(model = self.custom_model,
                      from_ = self.model,
                      system = self.system_prompt,
                      parameters={'temperature': self.temperature},
                      )

    def analyze_headline_sentiment(self, news_data: dict) -> dict:
        """Return a copy of the news dict with a 'scores' dict {ticker: score}."""
        company_names = news_data.get('company_names') or {}
        result = dict(news_data)
        if not company_names:
            result['scores'] = {}
            return result

        user_content = (
            f"company_names: {json.dumps(company_names, indent=2)}\n"
            f"message: {news_data.get('message', '')}"
        )
        messages = [{'role': 'user', 'content': user_content}]
        #if not self.system_prompt_in_model:
        #    messages.insert(0, {'role': 'system', 'content': self.system_prompt})

        try:
            response = ollama.chat(
                model=self.custom_model,
                messages=messages,
                format=build_scores_schema(list(company_names)),
                think=False,
                options={'temperature': 0.0},
                keep_alive=self.keep_alive,
            )
            content = response.get('message', {}).get('content', '').strip()
            if not content:
                raise RuntimeError(f"Ollama returned empty content: {response!r}")

            raw = json.loads(content)
            result['scores'] = {
                t: max(-1.0, min(1.0, float(raw.get(t, 0.0)))) for t in company_names
            }
            return result

        except ollama.ResponseError as e:
            raise RuntimeError(f"Ollama API error: {e.error}") from e

    def analyze_headlines(self, news_items: list[dict]) -> list[dict]:
        """Analyze every news item; return the batch with 'scores' added to each."""
        total = len(news_items)
        results = []
        for i, news in enumerate(news_items, start=1):
            print(f"analyzing news {i}/{total}: {news.get('headline', '')}")
            results.append(self.analyze_headline_sentiment(news))
        return results

def explode_scores(analized_list: list[dict]) -> pd.DataFrame:
    df_temp = pd.DataFrame(analized_list)
    df_temp['scores'] = df_temp['scores'].apply(lambda d: list(d.items()))
    df_temp = df_temp.explode('scores')
    df_temp[['ticker', 'score']] = pd.DataFrame(df_temp['scores'].tolist(), index=df_temp.index)
    return df_temp.reset_index()


def clean_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_temp = df_in.copy()
    df_temp['timestamp'] = pd.to_datetime(df_temp['updated_at'])
    df_temp['unique_id'] = df_temp['id'].astype(str) + '-' + df_temp['ticker']
    df_temp = df_temp[df_temp['score'].notna()]
    df_temp = df_temp[df_temp['score'] != 0]
    return df_temp[['id', 'unique_id', 'url', 'timestamp', 'ticker', 'score', 'message']]

In [31]:
fixed_articles = fix_news_list_from_raw(articles)
fixed_articles[0]

{'author': 'Benzinga Newsdesk',
 'content': '',
 'created_at': '2026-10-15T21:07:07Z',
 'headline': "'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive",
 'id': 62401077,
 'images': [],
 'source': 'benzinga',
 'summary': '',
 'symbols': ['AMZN', 'GOOG', 'GOOGL', 'MSFT'],
 'updated_at': '2026-10-15T21:07:07Z',
 'url': 'https://www.benzinga.com/news/26/10/62401077/investors-temper-anthropic-ipo-valuation-expectations-over-revenue-growth-slowdown-sources-say-reute',
 'message': "HEADLINE: 'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive\nSUMMARY: ",
 'company_names': {'AMZN': 'Amazon.com, Inc. Common Stock',
  'GOOG': 'Alphabet Inc. Class C Capital Stock',
  'GOOGL': 'Alphabet Inc. Class A Common Stock',
  'MSFT': 'Microsoft Corporation Common Stock'}}

In [32]:
with open('../../documentation/alpaca_news.csv', 'r') as file:
    news_df = pd.read_csv(file).rename(columns={'symbols': 'tickers'})
    news_list = news_df.to_dict(orient='records')
    json_data = json.dumps(news_list, indent=2)

In [33]:
news_list[0]

{'id': 62341492,
 'created_at': '2026-10-13T20:51:37Z',
 'headline': 'Applied Materials Stock Sinks for 6th Day, but Cantor Sees 85% Upside',
 'author': 'Piero Cingari',
 'tickers': "['AMAT', 'ASML', 'AVGO', 'KLAC', 'LRCX', 'MU', 'SKHY', 'SSNLF']",
 'source': 'benzinga',
 'summary': 'Applied Materials stock fell 7% on investor day, its sixth straight loss. Cantor Fitzgerald kept an $850 target, implying 85% upside.',
 'url': 'https://www.benzinga.com/analyst-stock-ratings/analyst-color/26/10/62341492/applied-materials-stock-sinks-sixth-day-cantor-850-target'}

In [34]:
#assets_us_equity

In [35]:
#assets_us_equity_df.head(3)
#news_list

In [36]:
#news_list = fix_news_list(news_list)

#retrieve company name from the assets_us_equity_df and add to news_list
for news in news_list:
    tickers = ast.literal_eval(news.get('tickers', []))
    #print(tickers, type(tickers))
    company_dict = {}
    for ticker in tickers:#here we want to extract a single string value from name
        print(ticker)
        try:
            name = assets_us_equity_df.loc[assets_us_equity_df['symbol'] == ticker, 'name'].values[0]
        except IndexError:
            name = None
        #name = assets_us_equity_df.loc[assets_us_equity_df['symbol'] == ticker, 'name'].values[0]
        print(ticker, name, type(name))
        company_dict[ticker] = name
    #company_names = list(company_dict.values())
    print(company_dict)
    message = ""# if summary is not nan then message is summary, else message is headline
    if pd.notna(news.get('summary')):
        message = news['summary']
    else:
        message = news.get('headline', '')

    news['message'] = message
    news['company_names'] = company_dict


In [37]:
#news_list[:2]

def build_scores_schema(tickers) -> dict:
    """JSON schema requiring one score in [-1, 1] for every ticker."""
    return {
        'type': 'object',
        'properties': {
            t: {'type': 'number', 'minimum': -1.0, 'maximum': 1.0} for t in tickers
        },
        'required': list(tickers),
        'additionalProperties': False,
    }

class HeadlineSentimentAnalyzer:
    def __init__(
        self,
        model: str = 'qwen3:8b',
        keep_alive: str = '10m',
        system_prompt_in_model: bool = False,
    ):
        self.model = model
        self.keep_alive = keep_alive
        self.system_prompt_in_model = system_prompt_in_model
        self.system_prompt = (
            "You are an expert financial analyst. Your task is to look at a news message "
            "and judge its sentiment regarding each of the given publicly traded companies. "
            "You will be given a JSON dictionary (company_names) that matches ticker symbols to company names, "
            "and the news message. "
            "Provide a rating score for every ticker strictly between -1.0 (very bearish/negative) and "
            "+1.0 (very bullish/positive). Highly neutral or purely factual announcements "
            "without clear financial impact, or companies not meaningfully discussed, should be given 0.0. "
            "Return a JSON object with the ticker symbols as keys and their sentiment scores as values."
        )

    def analyze_headline_sentiment(self, news_data: dict) -> dict:
        """Return a copy of the news dict with a 'scores' dict {ticker: score}."""
        company_names = news_data.get('company_names') or {}
        result = dict(news_data)
        if not company_names:
            result['scores'] = {}
            return result

        user_content = (
            f"company_names: {json.dumps(company_names, indent=2)}\n"
            f"message: {news_data.get('message', '')}"
        )
        messages = [{'role': 'user', 'content': user_content}]
        if not self.system_prompt_in_model:
            messages.insert(0, {'role': 'system', 'content': self.system_prompt})

        try:
            response = ollama.chat(
                model=self.model,
                messages=messages,
                format=build_scores_schema(list(company_names)),
                think=False,
                options={'temperature': 0.0},
                keep_alive=self.keep_alive,
            )
            content = response.get('message', {}).get('content', '').strip()
            if not content:
                raise RuntimeError(f"Ollama returned empty content: {response!r}")

            raw = json.loads(content)
            result['scores'] = {
                t: max(-1.0, min(1.0, float(raw.get(t, 0.0)))) for t in company_names
            }
            return result

        except ollama.ResponseError as e:
            raise RuntimeError(f"Ollama API error: {e.error}") from e

    def analyze_headlines(self, news_items: list[dict]) -> list[dict]:
        """Analyze every news item; return the batch with 'scores' added to each."""
        total = len(news_items)
        results = []
        for i, news in enumerate(news_items, start=1):
            print(f"analyzing news {i}/{total}: {news.get('headline', '')}")
            results.append(self.analyze_headline_sentiment(news))
        return results

In [38]:
hsa = HeadlineSentimentAnalyzer()
analized = hsa.analyze_headlines(fixed_articles)

analyzing news 1/372: 'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive
analyzing news 2/372: Google Rolls Out Universal Cart In U.S. Across Google Search, Gemini
analyzing news 3/372: 𝕏 Product Creator Allegra Jacchia Posts "We’re doubling down on video on 𝕏. Today, we're piloting Video Kickstart Bonus, an invite-only program that rewards creators for posting their video content here..."
analyzing news 4/372: Scotiabank Initiates Coverage On APA with Sector Perform Rating, Announces Price Target of $44
analyzing news 5/372: Lam Research Stock Hits Longest Losing Streak Ever: Are There Buyers Around?
analyzing news 6/372: Anthropic Needs Nearly $2.7B in Additional Monthly Revenue to Reach $100B Run Rate by Year-End
analyzing news 7/372: Google Quantum AI Achieves 40-Fold Error Reduction In Magic State Cultivation, Advancing Practical Quantum Computing
analyzing news 8/372: Google's $4 Billion Fuel-Cell Project Puts Bloo

In [39]:
fixed_articles[0]

{'author': 'Benzinga Newsdesk',
 'content': '',
 'created_at': '2026-10-15T21:07:07Z',
 'headline': "'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive",
 'id': 62401077,
 'images': [],
 'source': 'benzinga',
 'summary': '',
 'symbols': ['AMZN', 'GOOG', 'GOOGL', 'MSFT'],
 'updated_at': '2026-10-15T21:07:07Z',
 'url': 'https://www.benzinga.com/news/26/10/62401077/investors-temper-anthropic-ipo-valuation-expectations-over-revenue-growth-slowdown-sources-say-reute',
 'message': "HEADLINE: 'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive\nSUMMARY: ",
 'company_names': {'AMZN': 'Amazon.com, Inc. Common Stock',
  'GOOG': 'Alphabet Inc. Class C Capital Stock',
  'GOOGL': 'Alphabet Inc. Class A Common Stock',
  'MSFT': 'Microsoft Corporation Common Stock'}}

In [40]:
analized

[{'author': 'Benzinga Newsdesk',
  'content': '',
  'created_at': '2026-10-15T21:07:07Z',
  'headline': "'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive",
  'id': 62401077,
  'images': [],
  'source': 'benzinga',
  'summary': '',
  'symbols': ['AMZN', 'GOOG', 'GOOGL', 'MSFT'],
  'updated_at': '2026-10-15T21:07:07Z',
  'url': 'https://www.benzinga.com/news/26/10/62401077/investors-temper-anthropic-ipo-valuation-expectations-over-revenue-growth-slowdown-sources-say-reute',
  'message': "HEADLINE: 'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive\nSUMMARY: ",
  'company_names': {'AMZN': 'Amazon.com, Inc. Common Stock',
   'GOOG': 'Alphabet Inc. Class C Capital Stock',
   'GOOGL': 'Alphabet Inc. Class A Common Stock',
   'MSFT': 'Microsoft Corporation Common Stock'},
  'scores': {'AMZN': 0.0, 'GOOG': 0.0, 'GOOGL': 0.0, 'MSFT': 0.0}},
 {'author': 'Benzinga 

In [41]:
analized[0]

{'author': 'Benzinga Newsdesk',
 'content': '',
 'created_at': '2026-10-15T21:07:07Z',
 'headline': "'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive",
 'id': 62401077,
 'images': [],
 'source': 'benzinga',
 'summary': '',
 'symbols': ['AMZN', 'GOOG', 'GOOGL', 'MSFT'],
 'updated_at': '2026-10-15T21:07:07Z',
 'url': 'https://www.benzinga.com/news/26/10/62401077/investors-temper-anthropic-ipo-valuation-expectations-over-revenue-growth-slowdown-sources-say-reute',
 'message': "HEADLINE: 'Investors temper Anthropic IPO valuation expectations over revenue growth slowdown, sources say'- Reuters Exclusive\nSUMMARY: ",
 'company_names': {'AMZN': 'Amazon.com, Inc. Common Stock',
  'GOOG': 'Alphabet Inc. Class C Capital Stock',
  'GOOGL': 'Alphabet Inc. Class A Common Stock',
  'MSFT': 'Microsoft Corporation Common Stock'},
 'scores': {'AMZN': 0.0, 'GOOG': 0.0, 'GOOGL': 0.0, 'MSFT': 0.0}}

In [42]:
df_expanded = explode_scores(analized)
pd.unique(df_expanded['score'])

array([ 0.  ,  0.5 , -0.5 ,  0.8 ,  0.3 ,  0.1 ,  1.  , -0.75,  0.6 ,
       -1.  , -0.8 , -0.85, -0.7 , -0.3 ])

In [43]:


df_expanded_cleaned = clean_columns(df_expanded)

In [44]:
#create a column in df_expanded called timestamp that is a timestamp tyupe from updated_at
#df_expanded['timestamp'] = pd.to_datetime(df_expanded['updated_at'])
#df_expanded['unique_id'] = df_expanded['id'].astype(str) + '-' + df_expanded['ticker']
#df_expanded.info(verbose=True)
#df_expanded_cleaned = df_expanded[['id', 'unique_id', 'url', 'timestamp', 'ticker', 'score', 'message']]

In [45]:
#df_expanded[['id', 'unique_id', 'url', 'timestamp', 'ticker', 'score', 'message']]

In [46]:
df_expanded_cleaned.reset_index()

,index,id,unique_id,url,timestamp,ticker,score,message
0,9,62399951,62399951-APA,https://www.benzinga.com/news/26/10/62399951/s...,2026-10-15 20:26:40+00:00,APA,0.5,HEADLINE: Scotiabank Initiates Coverage On APA...
1,12,62398300,62398300-LRCX,https://www.benzinga.com/markets/tech/26/10/62...,2026-10-15 19:43:46+00:00,LRCX,-0.5,HEADLINE: Lam Research Stock Hits Longest Losi...
2,25,62395544,62395544-GOOG,https://www.benzinga.com/news/26/10/62395544/g...,2026-10-15 18:12:35+00:00,GOOG,0.8,HEADLINE: Google Quantum AI Achieves 40-Fold E...
3,26,62395544,62395544-GOOGL,https://www.benzinga.com/news/26/10/62395544/g...,2026-10-15 18:12:35+00:00,GOOGL,0.8,HEADLINE: Google Quantum AI Achieves 40-Fold E...
4,27,62395420,62395420-BE,https://www.benzinga.com/trading-ideas/long-id...,2026-10-15 18:08:14+00:00,BE,0.5,HEADLINE: Google's $4 Billion Fuel-Cell Projec...
...,...,...,...,...,...,...,...,...
376,809,62400669,62400669-WAT,https://www.benzinga.com/news/26/10/62400669/h...,2026-10-15 20:45:31+00:00,WAT,0.5,HEADLINE: Here’s How Much You Would Have Made ...
377,811,62386427,62386427-XEL,https://www.benzinga.com/news/26/10/62386427/b...,2026-10-15 14:40:59+00:00,XEL,0.5,HEADLINE: Barclays Maintains Overweight on Xce...
378,812,62386057,62386057-WDAY,https://www.benzinga.com/news/26/10/62386057/s...,2026-10-15 14:34:39+00:00,WDAY,0.3,"HEADLINE: Stifel Maintains Hold on Workday, Ra..."
379,813,62379021,62379021-WMT,https://www.benzinga.com/trading-ideas/movers/...,2026-10-15 12:33:52+00:00,WMT,0.8,HEADLINE: Why Is Walmart Stock Trading Higher ...


In [47]:
db = pgsql()
db.set_table_name('newstest')
db.set_pdf(df_expanded_cleaned)
db.write_pdf_to_sql()


True

In [48]:
df_expanded_cleaned

,id,unique_id,url,timestamp,ticker,score,message
9,62399951,62399951-APA,https://www.benzinga.com/news/26/10/62399951/s...,2026-10-15 20:26:40+00:00,APA,0.5,HEADLINE: Scotiabank Initiates Coverage On APA...
12,62398300,62398300-LRCX,https://www.benzinga.com/markets/tech/26/10/62...,2026-10-15 19:43:46+00:00,LRCX,-0.5,HEADLINE: Lam Research Stock Hits Longest Losi...
25,62395544,62395544-GOOG,https://www.benzinga.com/news/26/10/62395544/g...,2026-10-15 18:12:35+00:00,GOOG,0.8,HEADLINE: Google Quantum AI Achieves 40-Fold E...
26,62395544,62395544-GOOGL,https://www.benzinga.com/news/26/10/62395544/g...,2026-10-15 18:12:35+00:00,GOOGL,0.8,HEADLINE: Google Quantum AI Achieves 40-Fold E...
27,62395420,62395420-BE,https://www.benzinga.com/trading-ideas/long-id...,2026-10-15 18:08:14+00:00,BE,0.5,HEADLINE: Google's $4 Billion Fuel-Cell Projec...
...,...,...,...,...,...,...,...
809,62400669,62400669-WAT,https://www.benzinga.com/news/26/10/62400669/h...,2026-10-15 20:45:31+00:00,WAT,0.5,HEADLINE: Here’s How Much You Would Have Made ...
811,62386427,62386427-XEL,https://www.benzinga.com/news/26/10/62386427/b...,2026-10-15 14:40:59+00:00,XEL,0.5,HEADLINE: Barclays Maintains Overweight on Xce...
812,62386057,62386057-WDAY,https://www.benzinga.com/news/26/10/62386057/s...,2026-10-15 14:34:39+00:00,WDAY,0.3,"HEADLINE: Stifel Maintains Hold on Workday, Ra..."
813,62379021,62379021-WMT,https://www.benzinga.com/trading-ideas/movers/...,2026-10-15 12:33:52+00:00,WMT,0.8,HEADLINE: Why Is Walmart Stock Trading Higher ...
